## Inicio de una sesión PySpark

Para empezar, necesitaremos crear una sesión PySpark. No entraremos en mucho detalle aquí porque ya lo explicamos en la sesión anterior.

In [1]:
import pyspark
import os
from pyspark.sql import SparkSession

# Creamos una sesión PySpark contra nuestro servicio "spark-master"
spark = SparkSession.builder \
    .master(os.environ.get('SPARK_MASTER')) \
    .appName("pyspark-test") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/01 15:11:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Descarga de datos

A continuación, descargamos el fichero de "gran volumen de datos" de taxis FHV (For-Hire Vehicles) desde la web de [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).

In [2]:
!wget -nc https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2026-01.parquet

File ‘fhvhv_tripdata_2026-01.parquet’ already there; not retrieving.



## Lectura de datos

Ahora, vamos a leer los datos descargados. Como en este caso los datos los estamos descargando en formato Parquet, el código va a ser ligeramente diferente a lo que hicimos en la sesión anterior.

In [3]:
# Leemos el fichero Parquet
df = spark.read.parquet('fhvhv_tripdata_2026-01.parquet')

# Mostramos los primeros 3 registros
df.head(3)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B03404', originating_base_num='B03404', request_datetime=datetime.datetime(2026, 1, 1, 0, 50, 37), on_scene_datetime=datetime.datetime(2026, 1, 1, 0, 52, 31), pickup_datetime=datetime.datetime(2026, 1, 1, 0, 54, 30), dropoff_datetime=datetime.datetime(2026, 1, 1, 1, 13, 23), PULocationID=262, DOLocationID=79, trip_miles=4.3, trip_time=1133, base_passenger_fare=31.24, tolls=0.0, bcf=0.75, sales_tax=2.77, congestion_surcharge=2.75, airport_fee=0.0, tips=0.0, driver_pay=21.1, shared_request_flag='N', shared_match_flag='N', access_a_ride_flag='N', wav_request_flag='N', wav_match_flag='N', cbd_congestion_fee=1.5),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B03404', originating_base_num='B03404', request_datetime=datetime.datetime(2026, 1, 1, 0, 9, 12), on_scene_datetime=datetime.datetime(2026, 1, 1, 0, 12, 17), pickup_datetime=datetime.datetime(2026, 1, 1, 0, 12, 39), dropoff_datetime=datetime.datetime(2026, 1, 1, 0, 22, 42)

### Tipos de datos

Gracias a que los datos de los ficheros Parquet vienen tipados, las columnas con datos temporales tienen ya parseados como fechas. Para comprobarlo, podemos usar la propiedad `schema` de los _dataframes_ de PySpark.

In [4]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('originating_base_num', StringType(), True), StructField('request_datetime', TimestampNTZType(), True), StructField('on_scene_datetime', TimestampNTZType(), True), StructField('pickup_datetime', TimestampNTZType(), True), StructField('dropoff_datetime', TimestampNTZType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('trip_miles', DoubleType(), True), StructField('trip_time', LongType(), True), StructField('base_passenger_fare', DoubleType(), True), StructField('tolls', DoubleType(), True), StructField('bcf', DoubleType(), True), StructField('sales_tax', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('airport_fee', DoubleType(), True), StructField('tips', DoubleType(), True), StructField('driver_pay', DoubleType(), True), StructField('sha

### Proceso de ficheros en paralelo

En un proceso típico de Spark, tendríamos una cola de trabajos, por ejemplo ficheros a procesar, y la distribuiríamos de forma lo más equitativamente posible a través de los nodos de nuestro clúster Spark.

Sin embargo, en nuestro proceso actual tenemos un único fichero, de forma que no tenemos trabajor que distribuir. Para resolver esto, dividiremos el fichero en partes pequeñas. A estas partes, en Spark, las llamamos **particiones**. Para dividir nuestro _dataframe_ en partes, podemos usar `repartition`.

In [5]:
df.repartition(8)

DataFrame[hvfhs_license_num: string, dispatching_base_num: string, originating_base_num: string, request_datetime: timestamp_ntz, on_scene_datetime: timestamp_ntz, pickup_datetime: timestamp_ntz, dropoff_datetime: timestamp_ntz, PULocationID: int, DOLocationID: int, trip_miles: double, trip_time: bigint, base_passenger_fare: double, tolls: double, bcf: double, sales_tax: double, congestion_surcharge: double, airport_fee: double, tips: double, driver_pay: double, shared_request_flag: string, shared_match_flag: string, access_a_ride_flag: string, wav_request_flag: string, wav_match_flag: string, cbd_congestion_fee: double]

Ahora, para probar la paralelización, podemos simplemente guardar el _dataframe_ en una carpeta.

In [6]:
df.write.mode("overwrite").parquet('fhv/2026/01')

In [7]:
!ls -lah fhv/2026/01

total 556M
drwxr-xr-x 2 root root 4.0K Mar  1 15:13 .
drwxr-xr-x 3 root root 4.0K Mar  1 15:12 ..
-rw-r--r-- 1 root root  27M Mar  1 15:13 part-00000-65eb810c-7c04-4da2-97fc-f02f53ac111d-c000.snappy.parquet
-rw-r--r-- 1 root root 212K Mar  1 15:13 .part-00000-65eb810c-7c04-4da2-97fc-f02f53ac111d-c000.snappy.parquet.crc
-rw-r--r-- 1 root root  27M Mar  1 15:13 part-00001-65eb810c-7c04-4da2-97fc-f02f53ac111d-c000.snappy.parquet
-rw-r--r-- 1 root root 210K Mar  1 15:13 .part-00001-65eb810c-7c04-4da2-97fc-f02f53ac111d-c000.snappy.parquet.crc
-rw-r--r-- 1 root root  62M Mar  1 15:13 part-00002-65eb810c-7c04-4da2-97fc-f02f53ac111d-c000.snappy.parquet
-rw-r--r-- 1 root root 490K Mar  1 15:13 .part-00002-65eb810c-7c04-4da2-97fc-f02f53ac111d-c000.snappy.parquet.crc
-rw-r--r-- 1 root root  28M Mar  1 15:13 part-00003-65eb810c-7c04-4da2-97fc-f02f53ac111d-c000.snappy.parquet
-rw-r--r-- 1 root root 221K Mar  1 15:13 .part-00003-65eb810c-7c04-4da2-97fc-f02f53ac111d-c000.snappy.parquet.crc
-rw-r--r--

## Examinar el esquema

Aunque ya vimos una manera de examinar el esquema de un _dataframe_, Spark ofrece más alternativas. Un ejemplo es la función `printSchema`.

In [8]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_

### Seleccionar columnas y filtrar registros

Las dos principales utilidades que permiten reducir un _dataframe_ para ver solo algunas de sus columnas y filas son:

* `select`: que recibe los nombres de las columnas a mantener y devuelve un nuevo _dataframe_ con únicamente esas columnas,
* `filter`: que recibe una condición que se aplicará a todas las filas, devolviendo únicamente las que la satisfagan.

#### Transformaciones vs. acciones

La selección de columnas y el filtrado de filas, en Spark, son conocidas como **transformaciones** y su característica principal es que no son ejecutadas inmediatamente cuando son invocadas. En su lugar, ocurren más tarde, de forma "vaga".

In [16]:
# Aquí hemos definido un filtro, pero nada ha ocurrido aún con el _dataframe_
df_filtered = df \
    .select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
    .filter(df.hvfhs_license_num == 'HV0005')

Las **acciones**, a diferencia de las transformaciones, implican una carga inmediata de trabajo del lado de Spark. Un ejemplo son las funciones `head` y `show` que ya usamos antes. Usarlas implica la ejecución inmediata de una tarea, así como de las transformaciones que sean necesarias para completarla.

In [18]:
# Al lanzar show, la consulta de nuestro _dataframe_ disparará inmediatamente un trabajo en Spark
df_filtered.show()

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2026-01-01 00:09:37|2026-01-01 00:22:12|         141|         226|
|2026-01-01 00:26:40|2026-01-01 00:39:06|         226|         226|
|2026-01-01 00:45:34|2026-01-01 00:52:20|         260|          82|
|2026-01-01 00:56:27|2026-01-01 01:07:09|          82|          95|
|2026-01-01 00:25:27|2026-01-01 01:04:37|         144|         246|
|2026-01-01 00:54:32|2026-01-01 01:43:28|         148|         265|
|2026-01-01 00:14:33|2026-01-01 00:27:27|         226|         255|
|2026-01-01 00:31:15|2026-01-01 00:47:22|         255|          97|
|2026-01-01 00:50:58|2026-01-01 01:03:35|          65|         112|
|2026-01-01 00:12:31|2026-01-01 00:29:46|         169|         167|
|2026-01-01 00:32:15|2026-01-01 00:41:56|         167|          47|
|2026-01-01 00:48:12|2026-01-01 01:30:49|       

## Funciones

Hasta ahora hemos visto únicamente la parte más superficial de PySpark. De hecho, lo que hemos visto podría ser reemplazado por una consulta SQL bastante sencilla:

```sql
SELECT
    pickup_datetime,
    dropoff_datetime,
    PULocationID,
    DOLocationID
FROM [fhvhv_tripdata_2026-01]
WHERE
    hvfhs_license_num = 'HV0005'
```

Sin embargo, el punto fuerte de Spark está en la posibilidad de usar código para nuestras transformaciones, sea en forma de funciones predefinidas, o funciones definidas por el usuario.

### Funciones predefinidas

Las funciones predefinidas de PySpark están disponibles a través del objeto `functions` que de `pyspark.sql`:

In [19]:
from pyspark.sql import functions as F

Por convenio, suele importarse como `F` y la manera de usarlo es llamando a la función en el contexto de un `withColumn`. En este ejemplo, usamos `to_date` para ignorar la fecha y hora y quedarnos únicamente con el día de inicio y día de fin de cada trayecto.

In [23]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .select('pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

+-----------+------------+------------+------------+
|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-----------+------------+------------+------------+
| 2026-01-01|  2026-01-01|         262|          79|
| 2026-01-01|  2026-01-01|         195|          52|
| 2026-01-01|  2026-01-01|          25|         181|
| 2026-01-01|  2026-01-01|         141|         226|
| 2026-01-01|  2026-01-01|         226|         226|
| 2026-01-01|  2026-01-01|         260|          82|
| 2026-01-01|  2026-01-01|          82|          95|
| 2026-01-01|  2026-01-01|         144|         246|
| 2026-01-01|  2026-01-01|         162|         113|
| 2026-01-01|  2026-01-01|         148|         265|
| 2026-01-01|  2026-01-01|         226|         255|
| 2026-01-01|  2026-01-01|         255|          97|
| 2026-01-01|  2026-01-01|          65|         112|
| 2026-01-01|  2026-01-01|         169|         167|
| 2026-01-01|  2026-01-01|         167|          47|
| 2026-01-01|  2026-01-01|          78|       

### Funciones de usuario

Gracias a PySpark, podemos usar Python para definir nuestras propias funciones y usarlas para transformar grandes conjuntos de datos.

In [31]:
def duracion(inicio: datetime, fin: datetime) -> str:
    # Diferencia en minutos (valor absoluto por si vienen invertidos)
    minutos = abs((fin - inicio).total_seconds()) / 60

    if minutos < 5:
        return "duracion < 5 minutos"
    elif minutos < 10:
        return "5 minutos <= duracion < 10 minutos"
    elif minutos < 15:
        return "10 minutos <= duracion < 15 minutos"
    else:
        return ">= 15 minutos"

Una vez definida la función estándar de Python que queremos usar en PySpark, la "convertimos" en una función de usuario de PySpark usando `F.udf`:

In [32]:
from pyspark.sql import types

duracion_udf = F.udf(duracion, returnType=types.StringType())

Por fin, podemos usar nuestra función para realizar las transformaciones correspondientes.

In [35]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('duration_class', duracion_udf(df.pickup_datetime, df.dropoff_datetime)) \
    .select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID', 'duration_class') \
    .show()

[Stage 16:>                                                         (0 + 1) / 1]

+-------------------+-------------------+------------+------------+--------------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|      duration_class|
+-------------------+-------------------+------------+------------+--------------------+
|2026-01-01 00:54:30|2026-01-01 01:13:23|         262|          79|       >= 15 minutos|
|2026-01-01 00:12:39|2026-01-01 00:22:42|         195|          52|10 minutos <= dur...|
|2026-01-01 00:31:34|2026-01-01 00:55:21|          25|         181|       >= 15 minutos|
|2026-01-01 00:09:37|2026-01-01 00:22:12|         141|         226|10 minutos <= dur...|
|2026-01-01 00:26:40|2026-01-01 00:39:06|         226|         226|10 minutos <= dur...|
|2026-01-01 00:45:34|2026-01-01 00:52:20|         260|          82|5 minutos <= dura...|
|2026-01-01 00:56:27|2026-01-01 01:07:09|          82|          95|10 minutos <= dur...|
|2026-01-01 00:25:27|2026-01-01 01:04:37|         144|         246|       >= 15 minutos|
|2026-01-01 00:36:32|